In [1]:
import numpy as np
import pandas as pd

In [2]:
# Read data
YouGov = pd.read_csv(
    "../Raw data/australia.csv",
    na_values=[" ", "__NA__"],
    keep_default_na=True,
    low_memory=False
)

YouGov.shape

(53833, 513)

In [3]:
# Convert the endtime column to datetime format
YouGov["endtime"] = pd.to_datetime(
    YouGov["endtime"].str.split().str[0],
    format="%d/%m/%Y"
)

In [4]:
# Convert non-response categories to missing values
non_response_values = [
    "Prefer not to say",
    "Don't know",
    "NA",
    "N/A"
]

YouGov = YouGov.replace(non_response_values, np.nan)

In [5]:
# Drop columns with more than 11000 missing values
YouGov = YouGov.loc[:, YouGov.isna().sum() <= 11000]

YouGov.shape

(53833, 49)

In [6]:
# Remove all remaining rows with missing values
YouGov = YouGov.dropna()

YouGov.shape

(32658, 49)

In [7]:
# Create 2-week survey period index
start_date = YouGov["endtime"].min()
YouGov["week_number"] = ((YouGov["endtime"] - start_date).dt.days // 14) + 1

In [8]:
# Convert household size to numeric values
YouGov["household_size"] = YouGov["household_size"].map({
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8 or more": 8
})

YouGov = YouGov.dropna(subset=["household_size"])
YouGov.shape

(32658, 50)

In [9]:
# Convert agreement scale to numeric values
for col in ["r1_1", "r1_2"]:
    YouGov[col] = (
        YouGov[col]
        .astype(str)
        .str.extract(r"^([1-7])", expand=False)
        .astype(int)
    )

In [10]:
# Map frequency responses to numeric scale
for col in YouGov.columns:
    if col.startswith("i12_health_"):
        YouGov[col] = YouGov[col].map({
            "Always": 5,
            "Frequently": 4,
            "Sometimes": 3,
            "Rarely": 2,
            "Not at all": 1
        })

In [11]:
mask_items = ["i12_health_1", "i12_health_22", "i12_health_23", "i12_health_25"]
YouGov["face_mask_behaviour_scale"] = YouGov[mask_items].median(axis=1)

# Convert the face mask behaviour scale into a binary variable
YouGov["face_mask_behaviour_binary"] = (
    YouGov["face_mask_behaviour_scale"] >= 4
).astype(int)

In [12]:
# Collect all self-protective behaviour items
protective_behaviour_cols = []

for col in YouGov.columns:
    if col.startswith("i12_"):
        protective_behaviour_cols.append(col)

# Create the general protective behaviour scale
YouGov["protective_behaviour_scale"] = YouGov[protective_behaviour_cols].median(axis=1)

# Convert the general protective behaviour scale into a binary variable
YouGov["protective_behaviour_binary"] = (
    YouGov["protective_behaviour_scale"] >= 4
).astype(int)

In [13]:
# Combine comorbidity variables into a single categorical variable
d1_health_cols = []

for col in YouGov.columns:
    if col.startswith("d1_health_"):
        d1_health_cols.append(col)

YouGov["d1_comorbidities"] = "Yes"

if "d1_health_99" in YouGov.columns:
    YouGov.loc[YouGov["d1_health_99"] == "Yes", "d1_comorbidities"] = "No"

if "d1_health_98" in YouGov.columns:
    YouGov.loc[YouGov["d1_health_98"] == "Yes", "d1_comorbidities"] = np.nan

YouGov = YouGov.drop(d1_health_cols, axis=1)
YouGov = YouGov.dropna()

In [14]:
print(YouGov.shape)
print(YouGov.columns)

(32163, 40)
Index(['RecordNo', 'endtime', 'qweek', 'i2_health', 'i9_health', 'i11_health',
       'i12_health_1', 'i12_health_2', 'i12_health_3', 'i12_health_4',
       'i12_health_5', 'i12_health_6', 'i12_health_7', 'i12_health_8',
       'i12_health_11', 'i12_health_12', 'i12_health_13', 'i12_health_14',
       'i12_health_15', 'i12_health_16', 'weight', 'age', 'gender', 'state',
       'household_size', 'employment_status', 'WCRex2', 'cantril_ladder',
       'WCRex1', 'i12_health_22', 'i12_health_23', 'i12_health_25', 'r1_1',
       'r1_2', 'week_number', 'face_mask_behaviour_scale',
       'face_mask_behaviour_binary', 'protective_behaviour_scale',
       'protective_behaviour_binary', 'd1_comorbidities'],
      dtype='object')


In [15]:
YouGov = YouGov.drop(columns=["RecordNo", "qweek", "weight"])
YouGov = YouGov.drop(columns=protective_behaviour_cols)
YouGov = YouGov.drop(columns=["face_mask_behaviour_scale", "protective_behaviour_scale"])

print(YouGov.shape)
print(YouGov.columns)

(32163, 18)
Index(['endtime', 'i2_health', 'i9_health', 'i11_health', 'age', 'gender',
       'state', 'household_size', 'employment_status', 'WCRex2',
       'cantril_ladder', 'WCRex1', 'r1_1', 'r1_2', 'week_number',
       'face_mask_behaviour_binary', 'protective_behaviour_binary',
       'd1_comorbidities'],
      dtype='object')


In [16]:
YouGov.to_csv(
    "../Cleaned data/cleaned_YouGov_compare.csv",
    index=False
)

In [17]:
print(YouGov.shape)
YouGov.head(1)

(32163, 18)


,endtime,i2_health,i9_health,i11_health,age,gender,state,household_size,employment_status,WCRex2,cantril_ladder,WCRex1,r1_1,r1_2,week_number,face_mask_behaviour_binary,protective_behaviour_binary,d1_comorbidities
9023,2020-06-24,0.0,Not sure,Not sure,31,Male,Western Australia,1,Full time employment,Not very much confidence,8.0,Somewhat badly,5,1,1,0,0,Yes
